In [ ]:
import gc

try:
    import torch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print('GPU cache cleared.')
    else:
        print('CUDA not available, skip GPU cache clear.')
except Exception:
    gc.collect()
    print('Torch not available, skip GPU cache clear.')


# 第 5 课：Hybrid Search / 混合检索

前面你已经学过：
- bag-of-words 这类更偏字面匹配的检索
- embedding 这类更偏语义匹配的检索
- reranking

这一课我们进入真实 RAG 里非常常见的一步：

**Hybrid Search，也就是混合检索。**

它的核心想法非常朴素：

**字面匹配和语义匹配各有优缺点，那就把它们结合起来。**


## 1. 为什么不能只靠一种检索

如果只用字面匹配：
- 对关键词、专有名词很敏感
- 但对同义表达、改写不够鲁棒

如果只用 embedding 语义检索：
- 对语义改写更友好
- 但有时会漏掉一些关键词特别重要的场景

比如：
- 用户明确问一个错误码
- 一个产品型号
- 一个 API 名称
- 一个精确缩写

这时候纯语义检索不一定最稳。

所以真实系统里常常会说：

**既要 lexical matching，也要 semantic matching。**


## 2. 什么是 Hybrid Search

`Hybrid Search` 最常见的做法就是：

1. 先算一套稀疏检索分数
   比如 BM25 / 关键词匹配 / bag-of-words

2. 再算一套稠密检索分数
   比如 embedding 相似度

3. 最后把两套分数融合起来

融合方式有很多，但本质都一样：

**同时利用“字面证据”和“语义证据”。**


## 3. 这一课怎么讲 Hybrid Search

为了把主线讲清楚，这节课我们还是用教学版实现：

- 稀疏分数：词项重叠 / bag-of-words 风格分数
- 稠密分数：toy embedding 相似度
- 融合分数：加权组合

真实工业系统里当然会更复杂，但这已经足够让你彻底理解混合检索的思路。


In [ ]:
import math
import random
import re
from collections import Counter

random.seed(42)


In [ ]:
chunks = [
    'The API error code E401 usually means authentication failed due to an invalid token.',
    'Authentication problems often happen when access credentials are missing or expired.',
    'Chunking improves retrieval quality by splitting documents into focused passages.',
    'Embeddings make semantic retrieval possible by mapping text into vectors.',
    'Hybrid search combines lexical retrieval and semantic retrieval to improve recall and precision.'
]

for i, chunk in enumerate(chunks, start=1):
    print(f'chunk {i}: {chunk}')


## 4. 先定义稀疏检索分数

这部分代表 lexical matching，也就是更看重字面匹配。

这里我们用一个最简单的版本：
- query 和 chunk 的词项重叠数
- 再给某些完全匹配的词更多权重

虽然这不是 BM25，但已经能表达“字面检索”的核心味道。


In [ ]:
def tokenize(text):
    return re.findall(r"[a-zA-Z0-9]+", text.lower())


def lexical_score(query, chunk):
    q_tokens = tokenize(query)
    c_tokens = tokenize(chunk)
    c_counter = Counter(c_tokens)

    score = 0.0
    for token in q_tokens:
        if token in c_counter:
            # 对精确词匹配给基础分
            score += 1.0 * c_counter[token]

            # 对包含数字/错误码/专有字符串的 token 给更高权重
            if any(ch.isdigit() for ch in token):
                score += 2.0

    return score


## 5. 再定义稠密检索分数

这部分代表 semantic matching，也就是 embedding 风格检索。

我们继续沿用教学版 embedding：
- 每个 token 一个随机向量
- 文本向量 = token 向量平均
- 用余弦相似度打分


In [ ]:
token_to_vec = {}
embedding_dim = 8

def get_token_vector(token):
    if token not in token_to_vec:
        rnd = random.Random(hash(token) & 0xffffffff)
        token_to_vec[token] = [rnd.uniform(-1, 1) for _ in range(embedding_dim)]
    return token_to_vec[token]


def embed_text(text):
    tokens = tokenize(text)
    if not tokens:
        return [0.0] * embedding_dim
    vecs = [get_token_vector(token) for token in tokens]
    return [sum(v[i] for v in vecs) / len(vecs) for i in range(embedding_dim)]


def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)


chunk_embeddings = [embed_text(chunk) for chunk in chunks]

def dense_score(query, chunk_embedding):
    query_embedding = embed_text(query)
    return cosine_similarity(query_embedding, chunk_embedding)


## 6. 先分别看两种检索的结果

下面我们会分别跑：
- lexical retrieval
- dense retrieval

这样你能先看到它们各自偏向什么。


In [ ]:
query = 'What does error code E401 mean?'

lexical_results = []
dense_results = []

for chunk, chunk_emb in zip(chunks, chunk_embeddings):
    lexical_results.append((lexical_score(query, chunk), chunk))
    dense_results.append((dense_score(query, chunk_emb), chunk))

lexical_results.sort(key=lambda x: x[0], reverse=True)
dense_results.sort(key=lambda x: x[0], reverse=True)

print('query:', query)
print('\nLexical retrieval:')
for score, chunk in lexical_results:
    print(f'score={score:.3f}')
    print(chunk)
    print()

print('Dense retrieval:')
for score, chunk in dense_results:
    print(f'score={score:.3f}')
    print(chunk)
    print()


## 7. Hybrid Search：把两种分数融合起来

最简单的混合方式之一就是：

- 先把两种分数各自归一化
- 再按权重加起来

例如：

$hybrid = w_1 \cdot lexical + w_2 \cdot dense$

真实系统里还会有更复杂的融合策略，但第一课先把这个主线跑通。


In [ ]:
def min_max_normalize(scores):
    values = [score for score, _ in scores]
    min_v = min(values)
    max_v = max(values)
    if max_v == min_v:
        return [(0.0, item) for _, item in scores]
    return [((score - min_v) / (max_v - min_v), item) for score, item in scores]


lexical_norm = min_max_normalize(lexical_results)
dense_norm = min_max_normalize(dense_results)

lexical_map = {chunk: score for score, chunk in lexical_norm}
dense_map = {chunk: score for score, chunk in dense_norm}

alpha = 0.6  # lexical 权重
beta = 0.4   # dense 权重

hybrid_results = []
for chunk in chunks:
    score = alpha * lexical_map[chunk] + beta * dense_map[chunk]
    hybrid_results.append((score, chunk, lexical_map[chunk], dense_map[chunk]))

hybrid_results.sort(key=lambda x: x[0], reverse=True)

print('Hybrid retrieval:')
for hybrid_score, chunk, lexical_s, dense_s in hybrid_results:
    print(f'hybrid={hybrid_score:.3f} | lexical={lexical_s:.3f} | dense={dense_s:.3f}')
    print(chunk)
    print()


## 8. 为什么这个例子很典型

像 `E401` 这种 query 很有代表性，因为它包含：
- 明确错误码
- 明确字面词

这类问题里，lexical matching 往往特别重要。

但如果 query 换成更口语化的说法，比如：
- `Why am I getting an authentication failure?`

dense retrieval 可能就更有价值。

Hybrid Search 的意义就在这里：

**不同 query 类型里，让系统同时保留两边的优势。**


In [ ]:
query2 = 'Why am I getting an authentication failure?'

lexical_results_2 = []
dense_results_2 = []

for chunk, chunk_emb in zip(chunks, chunk_embeddings):
    lexical_results_2.append((lexical_score(query2, chunk), chunk))
    dense_results_2.append((dense_score(query2, chunk_emb), chunk))

lexical_results_2.sort(key=lambda x: x[0], reverse=True)
dense_results_2.sort(key=lambda x: x[0], reverse=True)

print('query:', query2)
print('\nLexical top-2:')
for score, chunk in lexical_results_2[:2]:
    print(f'score={score:.3f}')
    print(chunk)
    print()

print('Dense top-2:')
for score, chunk in dense_results_2[:2]:
    print(f'score={score:.3f}')
    print(chunk)
    print()


## 9. Hybrid Search 在真实系统里怎么落地

真实 RAG 系统里很常见的做法包括：

- `BM25 + Vector Search`
- 稀疏召回和稠密召回各取一批候选，再合并
- 合并后再交给 reranker

所以一个很常见的工程主线其实是：

1. Hybrid recall
2. Rerank
3. Final context selection
4. LLM generation


In [ ]:
gold_chunk = 'The API error code E401 usually means authentication failed due to an invalid token.'

lexical_rank = next(i for i, (_, chunk) in enumerate(lexical_results, start=1) if chunk == gold_chunk)
dense_rank = next(i for i, (_, chunk) in enumerate(dense_results, start=1) if chunk == gold_chunk)
hybrid_rank = next(i for i, (_, chunk, _, _) in enumerate(hybrid_results, start=1) if chunk == gold_chunk)

print('gold chunk 排名对比:')
print('lexical rank =', lexical_rank)
print('dense rank =', dense_rank)
print('hybrid rank =', hybrid_rank)


## 10. 这节课最该记住什么

如果你想抓住这节课的核心主线，就记住：

- lexical retrieval 擅长抓关键词和精确匹配
- dense retrieval 擅长抓语义相似
- hybrid search 的价值，就是把两者优势结合起来

所以真实 RAG 里很多时候不是“选哪一个”，而是：

**把两种信号都利用起来。**


## 11. 下一课最自然学什么

学完这一课后，最自然的下一步就是：

**RAG Evaluation 基础**

因为你现在已经看过：
- 检索
- chunking
- embedding
- reranking
- hybrid search

接下来最值得学的就是：这些环节到底怎么评估。
